Install and load all dependencies (first time only) \
NOTE: you may need to restart the runtime afterwards (CTRL+M .).

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os
os.chdir('/content/drive/MyDrive/policy')
print("Current Directory:", os.getcwd())  # Verify current directory

Current Directory: /content/drive/MyDrive/policy


Installing related libraries

In [4]:
!apt-get install -y \
    libgl1-mesa-dev \
    libgl1-mesa-glx \
    libglew-dev \
    libosmesa6-dev \
    software-properties-common
!pip install free-mujoco-py
!apt-get install -y patchelf
!pip install stable_baselines3
!pip install gym
!pip install shimmy
!pip install stable_baselines3
!pip install mujoco

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
software-properties-common is already the newest version (0.99.22.9).
The following additional packages will be installed:
  libegl-dev libgl-dev libgles-dev libgles1 libglu1-mesa libglu1-mesa-dev libglvnd-core-dev
  libglvnd-dev libglx-dev libopengl-dev libosmesa6
The following NEW packages will be installed:
  libegl-dev libgl-dev libgl1-mesa-dev libgl1-mesa-glx libgles-dev libgles1 libglew-dev
  libglu1-mesa libglu1-mesa-dev libglvnd-core-dev libglvnd-dev libglx-dev libopengl-dev libosmesa6
  libosmesa6-dev
0 upgraded, 15 newly installed, 0 to remove and 49 not upgraded.
Need to get 4,013 kB of archives.
After this operation, 19.4 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libglx-dev amd64 1.4.0-1 [14.1 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgl-dev amd64 1.4.0-1 [101 kB]
Get:3 http://archive.ubuntu.com/ubuntu 

In [5]:
from stable_baselines3.common.callbacks import EvalCallback
import matplotlib.pyplot as plt
from stable_baselines3 import PPO  # or SAC
from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy
import numpy as np
import gym
from IPython import display
import glob
import io
import base64
from IPython.display import HTML
from gym import wrappers
from IPython.display import clear_output
import gym
from env.custom_hopper import *
import torch.nn as nn
from stable_baselines3.common.vec_env import VecNormalize

Model instruction

In [ ]:
def train_and_evaluate(env_id, total_timesteps=2000000, n_eval_episodes=50, max_episode_steps=None):
    # Optionally set max episode steps

    if max_episode_steps:
        env = gym.make(env_id, max_episode_steps=max_episode_steps)
    else:
        env = gym.make(env_id)
        # Print environment info
    print(f'\nTraining on environment: {env_id}')
    print('State space:', env.observation_space)
    print('Action space:', env.action_space)
    print('Dynamics parameters:', env.get_parameters())
    '''
    model = SAC(
        "MlpPolicy",
        env,
        learning_rate=5e-4,
        buffer_size=100000,
        batch_size=64,
        learning_starts=1000,
        train_freq=(4, "step"),
        gradient_steps=1,
        tau=0.01,
        verbose=1
    )
    '''
    model = SAC(
        "MlpPolicy",
        env,
        learning_rate=5e-4,  # Slightly higher
        buffer_size=100000,  # Reduced
        batch_size=128,      # Smaller
        learning_starts=500, # Quicker start
        train_freq=(4, "step"),  # Less frequent updates
        gradient_steps=1,    # Fewer gradient steps
        tau=0.01,
        verbose=1
        )

    # Train the model
    print("\nStarting training...")
    model.learn(total_timesteps=total_timesteps)

    # Save the model
    model_path = f"hopper_{env_id.lower()}"
    model.save(model_path)
    print(f"\nModel saved to {model_path}")

    # Evaluate the model
    print("\nEvaluating model...")
    mean_reward, std_reward = evaluate_policy(
        model,
        env,
        n_eval_episodes=n_eval_episodes,
        deterministic=True
    )

    print(f"\nMean reward over {n_eval_episodes} episodes: {mean_reward:.2f} ± {std_reward:.2f}")

    return model, mean_reward, std_reward



/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Train model

In [ ]:
# Trained with SAC
def main():
    # Set random seed for reproducibility
    np.random.seed(42)
    env = gym.make('CustomHopper-source-v0')

    print('State space:', env.observation_space)  # state-space
    print('Action space:', env.action_space)  # action-space
    print('Dynamics parameters:', env.get_parameters())  # masses of each link of the Hopper

    # Train on source environment
    source_model, source_reward, source_std = train_and_evaluate(
        'CustomHopper-source-v0',
        total_timesteps=1000000,
        n_eval_episodes=50
    )
    '''
         # Train on target environment
    target_model, target_reward, target_std = train_and_evaluate(
         'CustomHopper-source-v0',
         total_timesteps=1000000,
         n_eval_episodes=50
    )
    '''

if __name__ == '__main__':
    main()

/usr/local/lib/python3.10/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


State space: Box(-inf, inf, (11,), float64)
Action space: Box(-1.0, 1.0, (3,), float32)
Dynamics parameters: [2.53429174 3.92699082 2.71433605 5.0893801 ]

Training on environment: CustomHopper-source-v0
State space: Box(-inf, inf, (11,), float64)
Action space: Box(-1.0, 1.0, (3,), float32)
Dynamics parameters: [2.53429174 3.92699082 2.71433605 5.0893801 ]
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.

Starting training...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 19.5     |
|    ep_rew_mean     | 10.8     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 1327     |
|    time_elapsed    | 0        |
|    total_timesteps | 78       |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 22.4     |
|    ep_rew_mean     | 19.6     |
| time/              |          |
|    episodes  

/usr/local/lib/python3.10/dist-packages/gym/utils/passive_env_checker.py:174: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  logger.warn(
/usr/local/lib/python3.10/dist-packages/gym/utils/passive_env_checker.py:190: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed `return_info` to return information from the environment resetting.
  logger.warn(
/usr/local/lib/python3.10/dist-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed `options` to allow the environment initialisation to be passed additional information.
  logger.warn(
/usr/local/lib/python3.10/dist-packages/gym/utils/passive_env_checker.py:227: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the envir

Streaming output truncated to the last 5000 lines.
|    ep_len_mean     | 443      |
|    ep_rew_mean     | 1.49e+03 |
| time/              |          |
|    episodes        | 1448     |
|    fps             | 160      |
|    time_elapsed    | 2731     |
|    total_timesteps | 439802   |
| train/             |          |
|    actor_loss      | -310     |
|    critic_loss     | 2.44     |
|    ent_coef        | 0.0443   |
|    ent_coef_loss   | 0.72     |
|    learning_rate   | 0.0005   |
|    n_updates       | 109825   |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 445      |
|    ep_rew_mean     | 1.49e+03 |
| time/              |          |
|    episodes        | 1452     |
|    fps             | 160      |
|    time_elapsed    | 2745     |
|    total_timesteps | 441802   |
| train/             |          |
|    actor_loss      | -314     |
|    critic_loss     | 3.06     |
|    ent_coef        | 0.0466  

/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(



Mean reward over 50 episodes: 1702.90 ± 18.16


In [7]:
# Helper function to show the rendered environment
def show_video(env_name, model, max_steps=2000):
    env = gym.make(env_name)
    env = gym.wrappers.RecordVideo(env, 'video', episode_trigger=lambda x: x == 0)

    obs = env.reset()
    done = False
    steps = 0
    while not done and steps < max_steps:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = env.step(action)
        steps += 1

    env.close()

    # Display the recorded video
    video_path = glob.glob("video/*.mp4")[0]
    video = io.open(video_path, 'r+b').read()
    encoded = base64.b64encode(video)
    return HTML(data='''
        <video width="360" height="360" controls>
            <source src="data:video/mp4;base64,{0}" type="video/mp4" />
        </video>'''.format(encoded.decode('ascii')))


Watch simulation

In [8]:

# Load the saved model
model_path = "/content/drive/MyDrive/policy/my_trained _models/hopper_customhopper-adr-v0.zip"  # or whatever your path is
loaded_model = SAC.load(model_path)

# Use it in show_video
show_video('CustomHopper-source-v0', loaded_model)

/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:95: UserWarning: You loaded a model that was trained using OpenAI Gym. We strongly recommend transitioning to Gymnasium by saving that model again.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/record_video.py:78: UserWarning: WARN: Overwriting existing videos at /content/drive/MyDrive/p

# Evaluation

evaluation result of trained models on different environ ments

In [11]:
# Load environment and model
source_env = gym.make('CustomHopper-source-v0')
target_env = gym.make('CustomHopper-target-v0')
source_model = SAC.load("/content/drive/MyDrive/policy/my_trained _models/hopper_customhopper-source-v0.zip")
target_model = SAC.load('/content/drive/MyDrive/policy/my_trained _models/hopper_customhopper-target-v0.zip')
udr_model = SAC.load('/content/drive/MyDrive/policy/my_trained _models/hopper_customhopper-udr-v0.zip')
adr_model = SAC.load('/content/drive/MyDrive/policy/my_trained _models/hopper_customhopper-adr-v0.zip')

/usr/local/lib/python3.10/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:95: UserWarning: You loaded a model that was trained using OpenAI Gym. We strongly recommend transitioning to Gymnasium by saving that model again.
  warnings.warn(


In [12]:
from stable_baselines3.common.evaluation import evaluate_policy

source_mean_reward, source_std_reward = evaluate_policy(source_model,source_env, n_eval_episodes=50,deterministic=True)  #evaluate source on source
target_mean_reward, target_std_reward = evaluate_policy(target_model, target_env,n_eval_episodes=50,deterministic=True)  #evaluate Target on target
sourceOnTarget_mean_reward, sourceOnTarget_std_reward = evaluate_policy(source_model, target_env, n_eval_episodes=50,deterministic=True)  #evaluate source on target
udrOnSource_mean_reward, udrOnSource_std_reward = evaluate_policy(udr_model, source_env, n_eval_episodes=50,deterministic=True) #evaluate udr on source
udronTarget_mean_reward,udronTarget_std_reward = evaluate_policy(udr_model, target_env, n_eval_episodes=50,deterministic=True) #evaluate udr on target
adrOnSource_mean_reward, adrOnSource_std_reward = evaluate_policy(adr_model, source_env, n_eval_episodes=50,deterministic=True) #evaluate adr on source
adronTarget_mean_reward,adronTarget_std_reward = evaluate_policy(adr_model, target_env, n_eval_episodes=50,deterministic=True) #evaluate adr on target

# Print results
print("\nSource Model on Source Domain Results:")
print(f"Mean Reward: {source_mean_reward:.2f} ± {source_std_reward:.2f}")

print("\nSource Model on Target Domain Results:")
print(f"Mean Reward: {sourceOnTarget_mean_reward:.2f} ± {sourceOnTarget_std_reward:.2f}")

print("\nTarget Model on Target Domain Results:")
print(f"Mean Reward: {target_mean_reward:.2f} ± {target_std_reward:.2f}")

print("\nUDR Model on Source Domain Results:")
print(f"Mean Reward: {udrOnSource_mean_reward:.2f} ± {udrOnSource_std_reward:.2f}")

print("\nUDR Model on Target Domain Results:")
print(f"Mean Reward: {udronTarget_mean_reward:.2f} ± {udronTarget_std_reward:.2f}")

print("\nADR Model on Source Domain Results:")
print(f"Mean Reward: {adrOnSource_mean_reward:.2f} ± {adrOnSource_std_reward:.2f}")

print("\nADR Model on Target Domain Results:")
print(f"Mean Reward: {adronTarget_mean_reward:.2f} ± {adronTarget_std_reward:.2f}")


/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/gym/utils/passive_env_checker.py:174: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  logger.warn(
/usr/local/lib/python3.10/dist-pa


Source Model on Source Domain Results:
Mean Reward: 1180.61 ± 48.50

Source Model on Target Domain Results:
Mean Reward: 1070.05 ± 5.64

Target Model on Target Domain Results:
Mean Reward: 1635.82 ± 9.20

UDR Model on Source Domain Results:
Mean Reward: 1731.05 ± 5.02

UDR Model on Target Domain Results:
Mean Reward: 1764.76 ± 5.90

ADR Model on Source Domain Results:
Mean Reward: 1687.36 ± 5.63

ADR Model on Target Domain Results:
Mean Reward: 1752.47 ± 34.90
